In [ ]:
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json

BUNDLE_NAME = "deployment01_20260928_v2.bin"
BUNDLE_SHA256 = "68f6c9eb2bc0ce12b2c9c6d89e33f22898fb201b33a792887f6c5967ad16ec4e"
ROOT = Path("/kaggle/working/deployment01-src")
OUT = Path("/kaggle/working/deployment01")
found = list(Path("/kaggle/input").rglob(BUNDLE_NAME))
if len(found) != 1:
    raise RuntimeError("Attach exactly one private Gate 3 bundle dataset.")
source = found[0]
if sha256(source.read_bytes()).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Wrong Gate 3 bundle SHA; do not extract it.")
if ROOT.exists() or OUT.exists():
    raise RuntimeError("Gate 3 output already exists. Use a fresh notebook/session; do not overwrite evidence.")
with ZipFile(source) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Gate 3 bundle CRC failed.")
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("Duplicate Gate 3 member.")
    manifest = json.loads(archive.read("gate3_bundle.json"))
    if manifest.get("schema") != "unified-kaggle-gate3-v1" or set(names) != set(manifest["files"]) | {"gate3_bundle.json"}:
        raise RuntimeError("Gate 3 bundle inventory differs.")
    ROOT.mkdir(parents=True)
    for name in names:
        target = (ROOT / name).resolve()
        if "\\" in name or ":" in name or not target.is_relative_to(ROOT.resolve()) or target.name in ("", ".", ".."):
            raise RuntimeError("Unsafe archive member.")
        raw = archive.read(name)
        if name != "gate3_bundle.json" and sha256(raw).hexdigest() != manifest["files"][name]:
            raise RuntimeError("Gate 3 member SHA differs.")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
print("Gate 3 bundle verified and extracted:", ROOT)
print("Members:", len(names), "SHA-256:", BUNDLE_SHA256)

# Check required Secret before dependency setup. Never print its value.
from kaggle_secrets import UserSecretsClient
assert len(UserSecretsClient().get_secret("AI_REMOTE_API_KEY") or "") >= 24, "Enable AI_REMOTE_API_KEY Secret"
print("Required API key attached. Run Cell 2 next.")


In [ ]:
import subprocess, sys
print("Starting exact Gate 2 environment, one Base, unified server and one tunnel. This can take several minutes.", flush=True)
result = subprocess.run([sys.executable, str(ROOT / "scripts/unified_gate3_bootstrap.py"), "--output", str(OUT)], cwd=ROOT)
if result.returncode:
    print("Setup stopped. Open /kaggle/working/deployment01/startup.json and the named stage log; do not start another service.")
else:
    print("Unified server is running. Keep this notebook session active while testing the application.")


In [ ]:
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
report = json.loads((OUT / "startup.json").read_text())
print("Setup status:", report["status"], "phase:", report["phase"])
print("Base load count:", report.get("public_health", {}).get("foundation_load_count"))
print("Public health:", report.get("public_health"))
health = report.get("public_health", {})
assert report["status"] == "READY_FOR_LIVE_ACCEPTANCE", "Setup did not become ready"
assert health.get("gpu_runtime_ready") and health.get("foundation_load_count") == 1, "Worker is not ready"
assert health.get("diagnostics_version") == "deployment-01", "Wrong worker version"
if (OUT / "endpoint.json").is_file():
    endpoint = json.loads((OUT / "endpoint.json").read_text())
    print("ONE URL:", endpoint["url"])
    print("Use the same private AI_REMOTE_API_KEY Kaggle Secret in backend/.env. Never paste its value here.")
allow = ["startup.json", "model.json", "endpoint.json", "environment.log", "artifact_preflight.log", "torch_constraints.txt"]
with ZipFile(OUT / "evidence.zip", "x", ZIP_DEFLATED) as archive:
    for name in allow:
        path = OUT / name
        if path.is_file():
            archive.write(path, name)
print("Download:", OUT / "evidence.zip")
print("DEPLOYMENT_01_SERVICE_READY, fresh application rehearsal still required.")


In [ ]:
# Run this cell only AFTER the live application requests finish.
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from zipfile import ZipFile, ZIP_DEFLATED
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("AI_REMOTE_API_KEY")
if not key:
    raise RuntimeError("Attach the AI_REMOTE_API_KEY Secret first.")
base = "http://127.0.0.1:8765"
def status():
    with urlopen(Request(base + "/status", headers={"X-API-Key": key}), timeout=20) as response:
        return json.load(response)
before = status()
boundary = "gate3-safe-probe-boundary"
def probe(feature, style, auth):
    body = (f"--{boundary}\r\nContent-Disposition: form-data; name=\"style_id\"\r\n\r\n{style}\r\n"
            f"--{boundary}\r\nContent-Disposition: form-data; name=\"image\"; filename=\"bad.png\"\r\n"
            "Content-Type: image/png\r\n\r\nBAD\r\n"
            f"--{boundary}--\r\n").encode()
    request = Request(base + f"/{feature}/generate", data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}", "X-API-Key": auth}, method="POST")
    try:
        with urlopen(request, timeout=20) as response:
            return response.status
    except HTTPError as error:
        return error.code
probes = {
    "wrong_key": probe("hairstyle", "crew_cut", "wrong"),
    "unknown_feature": probe("unknown", "crew_cut", key),
    "invalid_style": probe("hairstyle", "invalid_style", key),
    "invalid_image": probe("hairstyle", "crew_cut", key),
}
assert probes == {"wrong_key": 401, "unknown_feature": 404, "invalid_style": 400, "invalid_image": 400}, probes
after = status()
assert after["gpu_runtime_ready"] and after["foundation_load_count"] == 1
assert len(after["requests"]) == len(before["requests"]), "Safe HTTP errors unexpectedly generated an image"
(OUT / "http_probes.json").write_text(json.dumps(probes, indent=2) + "\n")
(OUT / "live_status.json").write_text(json.dumps(after, indent=2) + "\n")
print("Safe HTTP error statuses:", probes)
print("Base loads:", after["foundation_load_count"])
print("Recorded successful GPU requests:", [(r["feature"], r["style_id"], r["adapter_id"], r["http_status"]) for r in after["requests"]])
print("Switch events:", after["switch_events"])
(OUT / "deployment01_sources.json").write_bytes((ROOT / "deployment01_sources.json").read_bytes())
with ZipFile(OUT / "live-evidence.zip", "x", ZIP_DEFLATED) as archive:
    for name in ("startup.json", "model.json", "endpoint.json", "live_status.json", "http_probes.json", "environment.log", "artifact_preflight.log", "server.log", "deployment01_sources.json"):
        file = OUT / name
        if file.is_file():
            archive.write(file, name)
print("Download and return:", OUT / "live-evidence.zip")

print("DEPLOYMENT_01_REHEARSAL_REVIEW_REQUIRED, no automatic deployment pass.")
